# Phase 3 — Preference Generation Report

Reads `data/preferences/dpo_report.jsonl` (every prompt processed — pairs generated and discarded) and breaks down the preference-pair generation process. This is the evidence that the **Execution-Based Iterative DPO** pipeline works: how many pairs survived, which case dominated (A/B/C), and whether the composite reward actually separates chosen from rejected on quality metrics.

**Case classification:**
- **Case A** — one candidate passes execution, the other fails → clear winner
- **Case B** — both fail → discarded (prevents noisy gradients)
- **Case C** — both pass → ranked by composite quality metric (CC + lint)
- **Case C_tied** — both pass with identical composite scores → discarded

Dependencies (`pandas`, `matplotlib`, `seaborn`) are in the `dev` dependency group in `pyproject.toml` — run `uv sync` rather than `pip install`ing them inline, so the notebook's environment stays in sync with the rest of the project.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.preference_generation.preference_generation_config import DPO_REPORT

sys.path.insert(0, str(Path.cwd().parent))

records = []
with open(DPO_REPORT, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
total = len(df)
generated = (~df["discarded"]).sum()
discarded = df["discarded"].sum()

print(f"Prompts processed: {total}")
print(f"Pairs generated:   {generated} ({generated / total:.1%})")
print(f"Pairs discarded:   {discarded} ({discarded / total:.1%})")

## Case distribution (A / B / C)

The ratio between cases reveals how useful the composite reward is:
- If **Case C > 15%**, the composite metric is actively separating quality among working code — this is the core contribution.
- If **Case B** dominates, the SFT model is still generating too much broken code — may need more SFT epochs.
- If **Case A** dominates, most signal comes from the binary pass/fail — the composite adds less value.

In [ ]:
case_counts = df["case"].value_counts()
case_order = ["A", "B", "C", "C_tied"]
case_colors = {"A": "#4CAF50", "B": "#F44336", "C": "#2196F3", "C_tied": "#FF9800"}

existing_cases = [c for c in case_order if c in case_counts.index]
colors = [case_colors[c] for c in existing_cases]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(existing_cases, [case_counts[c] for c in existing_cases], color=colors)
axes[0].set_title("Case distribution")
axes[0].set_ylabel("Count")
for i, c in enumerate(existing_cases):
    axes[0].text(i, case_counts[c] + 0.5, str(case_counts[c]), ha="center")

# Pie chart
axes[1].pie(
    [case_counts[c] for c in existing_cases],
    labels=existing_cases,
    autopct="%1.1f%%",
    colors=colors,
)
axes[1].set_title("Case distribution (%)")

plt.tight_layout()
plt.show()

## Composite score: chosen vs. rejected

If the composite reward is working, chosen samples should consistently score higher than rejected samples. The gap between distributions is the "preference margin" that DPO will learn from.

In [ ]:
df_valid = df[~df["discarded"]].copy()

if len(df_valid) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_valid["chosen_score"], bins=25, alpha=0.6, label="Chosen", color="#4CAF50")
    ax.hist(df_valid["rejected_score"], bins=25, alpha=0.6, label="Rejected", color="#F44336")
    ax.set_title("Composite score distribution — chosen vs. rejected")
    ax.set_xlabel("Composite score")
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Chosen  — mean: {df_valid['chosen_score'].mean():.4f}, std: {df_valid['chosen_score'].std():.4f}")
    print(f"Rejected — mean: {df_valid['rejected_score'].mean():.4f}, std: {df_valid['rejected_score'].std():.4f}")
else:
    print("No valid pairs generated — run Phase 3 first.")

## Cyclomatic complexity: chosen vs. rejected

**This is the star graph of the project.** If chosen code has *lower* CC than rejected code, the composite reward is actively preventing spaghetti code — the core hypothesis validated empirically.

In [ ]:
if len(df_valid) > 0:
    cc_data = pd.DataFrame({
        "Chosen": df_valid["chosen_complexity"],
        "Rejected": df_valid["rejected_complexity"],
    }).dropna()

    fig, ax = plt.subplots(figsize=(8, 5))
    cc_melted = cc_data.melt(var_name="Type", value_name="Cyclomatic Complexity")
    sns.boxplot(data=cc_melted, x="Type", y="Cyclomatic Complexity",
                palette={"Chosen": "#4CAF50", "Rejected": "#F44336"}, ax=ax)
    ax.set_title("Cyclomatic complexity — chosen vs. rejected")
    plt.tight_layout()
    plt.show()

    print(f"Chosen  — median CC: {df_valid['chosen_complexity'].median():.1f}")
    print(f"Rejected — median CC: {df_valid['rejected_complexity'].median():.1f}")
else:
    print("No valid pairs generated — run Phase 3 first.")

## Lint errors: chosen vs. rejected

Fewer lint errors in chosen code means the model is being steered toward cleaner, more idiomatic style — the $R_{\text{style}}$ component of the composite reward.

In [ ]:
if len(df_valid) > 0:
    lint_data = pd.DataFrame({
        "Chosen": df_valid["chosen_lint_errors"],
        "Rejected": df_valid["rejected_lint_errors"],
    }).dropna()

    fig, ax = plt.subplots(figsize=(8, 5))
    lint_melted = lint_data.melt(var_name="Type", value_name="Lint Errors")
    sns.boxplot(data=lint_melted, x="Type", y="Lint Errors",
                palette={"Chosen": "#4CAF50", "Rejected": "#F44336"}, ax=ax)
    ax.set_title("Lint errors — chosen vs. rejected")
    plt.tight_layout()
    plt.show()

    print(f"Chosen  — median lint errors: {df_valid['chosen_lint_errors'].median():.1f}")
    print(f"Rejected — median lint errors: {df_valid['rejected_lint_errors'].median():.1f}")
else:
    print("No valid pairs generated — run Phase 3 first.")

## Case distribution by language

Some languages may have higher Case B rates (both fail) due to weaker SFT coverage or sandbox limitations (e.g. missing toolchain). A language with disproportionately high B rates is a signal to investigate.

In [ ]:
if "language" in df.columns:
    ct = pd.crosstab(df["language"], df["case"], normalize="index")
    ct = ct.reindex(columns=[c for c in ["A", "B", "C", "C_tied"] if c in ct.columns])
    ct.plot(kind="bar", stacked=True, figsize=(10, 5),
            color=[case_colors.get(c, "#999") for c in ct.columns])
    plt.title("Case distribution by language")
    plt.ylabel("Proportion")
    plt.xticks(rotation=20)
    plt.legend(title="Case")
    plt.tight_layout()
    plt.show()
else:
    print("No language column found.")

## Qualitative samples

A few representative chosen / rejected pairs to inspect manually. Focus on Case C pairs (both passed, ranked by quality) — these are where the composite reward is making the decision.

In [ ]:
case_c_pairs = df_valid[df_valid["case"] == "C"]
sample_n = min(3, len(case_c_pairs))

if sample_n > 0:
    for _, row in case_c_pairs.sample(sample_n, random_state=42).iterrows():
        print(f"{'=' * 70}")
        print(f"Prompt: {row['prompt'][:120]}...")
        print(f"Language: {row['language']}")
        print(f"Chosen score: {row['chosen_score']:.4f} | CC: {row['chosen_complexity']} | Lint: {row['chosen_lint_errors']}")
        print(f"Rejected score: {row['rejected_score']:.4f} | CC: {row['rejected_complexity']} | Lint: {row['rejected_lint_errors']}")
        print(f"\n--- CHOSEN ---\n{row['chosen'][:300]}")
        print(f"\n--- REJECTED ---\n{row['rejected'][:300]}\n")
else:
    print("No Case C pairs found — all preference signal came from pass/fail (Case A).")
    if len(df_valid) > 0:
        print("Showing Case A samples instead:")
        for _, row in df_valid.head(3).iterrows():
            print(f"\nPrompt: {row['prompt'][:120]}...")
            print(f"Chosen score: {row['chosen_score']:.4f}")
            print(f"Rejected score: {row['rejected_score']:.4f}")

## Key takeaways

- **Case distribution:** [fill after run — was Case C > 15%?]
- **Composite reward effectiveness:** [does chosen have lower CC and fewer lint errors than rejected?]
- **Language coverage:** [any language with disproportionate Case B?]
- **Next step:** Proceed to Phase 4 (DPO training) using `data/preferences/dpo_dataset.jsonl`